In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
#Text processing - 1 converting into lowercase
df["review"] = df["review"].str.lower()
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [5]:
# 2 - removing URLs
import re
def remove_url(text):
    text =re.sub("http/S+","",text) #(pattern , repl , string)
    return text
    
df["review"] = df["review"].apply(remove_url) 

In [6]:
# remove punctuation
def remove_punct(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"]= df["review"].apply(remove_punct)


In [7]:
# remove html tags 
def remove_html(text):
    text = re.sub(r"<.*?>","",text)
    return text
df["review"]  = df["review"].apply(remove_html)
df.sample(10)

,review,sentiment
18346,even if voskhozhdeniye was your favorite film ...,positive
23445,im a big fan of naruto even though i havent wa...,positive
8636,wow this was probably the worst dcom ever i wa...,negative
7301,excuse me but my tongue was too in my cheek wh...,negative
15045,theres no better way to describe the horrendou...,negative
6989,this stirring western spins the tale of the fa...,positive
11074,this is a great tv movie with a good story and...,positive
47058,i wont try to speculate as to what brando was ...,positive
23646,dont let the premise fool youthis was one funn...,negative
28358,extremely thin plot of satanic rituals or some...,negative


In [8]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\Murli
[nltk_data]     Bhatt\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Murli
[nltk_data]     Bhatt\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Murli
[nltk_data]     Bhatt\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
# remove stop words 

def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    filtered_words = []
    tokens = word_tokenize(text)

    for word in tokens:
        if word not in stop_words:
            filtered_words.append(word)
    
    # This space " " is the most important character in your preprocessing!
    return " ".join(filtered_words)
            

In [10]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [11]:
# Stemming 
from nltk.stem import PorterStemmer
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)
df["review"] = df["review"].apply(stemming)
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the film techniqu...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love in the time of money is a v...,positive


In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

y = df["sentiment"]
y.head()


0    1
1    1
2    1
3    0
4    1
Name: sentiment, dtype: int64

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features = 5000)

X = tfidf.fit_transform(df["review"])

In [14]:
from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test =  train_test_split(X,y,test_size = 0.2,random_state=42)

x_train.shape


(40000, 5000)

In [17]:
# build model 
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset , DataLoader

x_train = x_train.toarray()
x_test  = x_test.toarray()



In [18]:
train_set = TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [19]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [20]:
import torch.optim as optim


In [21]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [22]:
input_size = x_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [23]:
# Training OF RNN model

epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.33086180686950684
epoch = 2/10 and loss = 0.4334987699985504
epoch = 3/10 and loss = 0.3049369156360626
epoch = 4/10 and loss = 0.29382944107055664
epoch = 5/10 and loss = 0.13755309581756592
epoch = 6/10 and loss = 0.18070504069328308
epoch = 7/10 and loss = 0.18471035361289978
epoch = 8/10 and loss = 0.16784459352493286
epoch = 9/10 and loss = 0.10034479200839996
epoch = 10/10 and loss = 0.17491517961025238


In [24]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 87.96000000000001
